In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

/kaggle/input/playground-series-s4e11/sample_submission.csv
/kaggle/input/playground-series-s4e11/train.csv
/kaggle/input/playground-series-s4e11/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/playground-series-s4e11/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s4e11/test.csv', index_col='id')  
sub = pd.read_csv('/kaggle/input/playground-series-s4e11/sample_submission.csv')

In [3]:
X= train.drop('Depression', axis=1)
y = train['Depression']

In [4]:
all_data = pd.concat([X, test], axis=0)

all_data_numerical = all_data[['Age','Academic Pressure','Work Pressure','CGPA','Study Satisfaction',	'Job Satisfaction',	'Sleep Duration','Work/Study Hours','Financial Stress']]
all_data_categorical = all_data.drop(columns=['Age','Academic Pressure','Work Pressure','CGPA','Study Satisfaction',	'Job Satisfaction',	'Sleep Duration','Work/Study Hours','Financial Stress'], axis=1)

SD_map = {'More than 8 hours': 9, 'Less than 5 hours':4, '5-6 hours':5.5, '7-8 hours':7.5,
       'Sleep_Duration':0, '1-2 hours':1.5, '6-8 hours':7, '4-6 hours':5,
       '6-7 hours':6.5, '10-11 hours':10.5, '8-9 hours':8.5, '40-45 hours':0,
       '9-11 hours':10, '2-3 hours':2.5, '3-4 hours':3.5, 'Moderate':0, '55-66 hours':0,
       '4-5 hours':4.5, '9-6 hours':0, '1-3 hours':2, 'Indore':0, '45':0, '1-6 hours':0,
       '35-36 hours':0, '8 hours':8, 'No':0, '10-6 hours':8, 'than 5 hours':0,
       '49 hours':0, 'Unhealthy':0, 'Work_Study_Hours':0, '3-6 hours':4.5,
       '45-48 hours':0, '9-5':8, 'Pune':0, '9-5 hours':8}

yes_no_map = {'Yes':1, 'No':0}

male_female_map = {'Male':1, 'Female':0}  

w_or_s_map = {'Working Professional':1, 'Student':0}


from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
le_profession = LabelEncoder()
le_dietary = LabelEncoder()
le_degree = LabelEncoder()

all_data_numerical['sleep_time'] = all_data_numerical['Sleep Duration'].map(SD_map)
all_data_numerical = all_data_numerical.drop(columns=['Sleep Duration'], axis=1)

all_data_numerical['pressure'] = all_data_numerical[['Academic Pressure','Work Pressure']].sum(axis=1)
all_data_numerical = all_data_numerical.drop(columns=['Academic Pressure','Work Pressure'], axis=1)

all_data_numerical['sutisfaction'] = all_data_numerical[['Study Satisfaction',	'Job Satisfaction']].sum(axis=1)
all_data_numerical = all_data_numerical.drop(columns=['Study Satisfaction',	'Job Satisfaction'], axis=1)




# Fill NaN values

all_data_numerical['CGPA'].fillna(0, inplace=True)
all_data_numerical['Financial Stress'].fillna(0, inplace=True)


all_data_categorical.drop('Name', axis = 1, inplace = True)
all_data_categorical['Have you ever had suicidal thoughts ?'] = all_data_categorical['Have you ever had suicidal thoughts ?'].map(yes_no_map)
all_data_categorical['Family History of Mental Illness'] = all_data_categorical['Family History of Mental Illness'].map(yes_no_map)

all_data_categorical['Profession'].fillna(all_data_categorical['Working Professional or Student'], inplace=True)
all_data_categorical['Gender'] = all_data_categorical['Gender'].map(male_female_map)


all_data_categorical['Working Professional or Student'] = all_data_categorical['Working Professional or Student'].map(w_or_s_map)



# Fill NaN for categorical features with less than or equal to 3 unique values

for col in all_data_categorical.columns:
    if len(all_data_categorical[col].value_counts()) <= 3:
        all_data_categorical[col] = all_data_categorical[col].fillna('missing')

# Apply Label Encoding

all_data_categorical['City'] = le_city.fit_transform(all_data_categorical['City'])
all_data_categorical['Profession'] = le_profession.fit_transform(all_data_categorical['Profession'])
all_data_categorical['Dietary Habits'] = le_dietary.fit_transform(all_data_categorical['Dietary Habits'])
all_data_categorical['Degree'] = le_degree.fit_transform(all_data_categorical['Degree'])
all_data_categorical = all_data_categorical.astype('category')

all_data_all = pd.concat([all_data_numerical, all_data_categorical], axis=1)

/tmp/ipykernel_17/3449818642.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_data_numerical['sleep_time'] = all_data_numerical['Sleep Duration'].map(SD_map)
/tmp/ipykernel_17/3449818642.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  all_data_numerical['CGPA'].fillna(0, inp

In [5]:
X = all_data_all[:len(X)]
test = all_data_all[len(X):]
y = y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier, Pool

In [7]:

# Define the parameter grid for CatBoost
param_grid_cat = {
    'iterations': [100, 200],
    'learning_rate': [0.01, 0.1],
    'depth': [6, 8],
}

# Initialize the CatBoost classifier
model_cat = CatBoostClassifier(loss_function='Logloss', eval_metric='Accuracy', random_seed=42, verbose=10)

# Initialize GridSearchCV
grid_search_cat = GridSearchCV(estimator=model_cat, param_grid=param_grid_cat, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the data
grid_search_cat.fit(X_train, y_train, cat_features=list(all_data_categorical.columns)) # Use cat_features

# Print the best parameters and score
print(f"Best parameters (CatBoost): {grid_search_cat.best_params_}")
print(f"Best cross-validation score (CatBoost): {grid_search_cat.best_score_}")

# Make predictions using the best model
y_pred_cat = grid_search_cat.best_estimator_.predict(X_test)
acc_cat = accuracy_score(y_test, y_pred_cat)
print(f"Accuracy on test set (CatBoost): {acc_cat}")

0:	learn: 0.9167888	total: 207ms	remaining: 20.5s
10:	learn: 0.9267946	total: 1.61s	remaining: 13s
20:	learn: 0.9280939	total: 2.89s	remaining: 10.9s
30:	learn: 0.9282050	total: 4.1s	remaining: 9.13s
40:	learn: 0.9283937	total: 5.44s	remaining: 7.83s
50:	learn: 0.9285048	total: 7.16s	remaining: 6.88s
60:	learn: 0.9285492	total: 8.88s	remaining: 5.68s
70:	learn: 0.9289823	total: 10.1s	remaining: 4.14s
80:	learn: 0.9295376	total: 11.4s	remaining: 2.67s
90:	learn: 0.9298263	total: 12.8s	remaining: 1.26s
99:	learn: 0.9300484	total: 13.9s	remaining: 0us
0:	learn: 0.9173885	total: 147ms	remaining: 14.5s
10:	learn: 0.9272388	total: 1.48s	remaining: 12s
20:	learn: 0.9282383	total: 2.78s	remaining: 10.4s
30:	learn: 0.9285714	total: 4.05s	remaining: 9.03s
40:	learn: 0.9285159	total: 5.24s	remaining: 7.55s
50:	learn: 0.9291600	total: 6.55s	remaining: 6.29s
60:	learn: 0.9292155	total: 7.8s	remaining: 4.99s
70:	learn: 0.9296375	total: 9.13s	remaining: 3.73s
80:	learn: 0.9302372	total: 10.5s	remaini

/opt/conda/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


0:	learn: 0.9164001	total: 242ms	remaining: 24s
10:	learn: 0.9269612	total: 1.59s	remaining: 12.8s
20:	learn: 0.9275386	total: 2.89s	remaining: 10.9s
30:	learn: 0.9283382	total: 4.15s	remaining: 9.25s
40:	learn: 0.9283160	total: 5.54s	remaining: 7.98s
50:	learn: 0.9284493	total: 7.26s	remaining: 6.97s
60:	learn: 0.9286825	total: 8.99s	remaining: 5.75s
70:	learn: 0.9290712	total: 10.3s	remaining: 4.22s
80:	learn: 0.9297486	total: 11.7s	remaining: 2.74s
90:	learn: 0.9299263	total: 13s	remaining: 1.29s
99:	learn: 0.9300817	total: 14.2s	remaining: 0us
0:	learn: 0.9164001	total: 143ms	remaining: 14.2s
10:	learn: 0.9296153	total: 1.37s	remaining: 11.1s
20:	learn: 0.9330912	total: 2.81s	remaining: 10.6s
30:	learn: 0.9352345	total: 4.36s	remaining: 9.7s
40:	learn: 0.9367004	total: 5.6s	remaining: 8.06s
50:	learn: 0.9374778	total: 6.94s	remaining: 6.67s
60:	learn: 0.9379109	total: 8.29s	remaining: 5.3s
70:	learn: 0.9385550	total: 9.67s	remaining: 3.95s
80:	learn: 0.9389659	total: 10.9s	remainin

In [8]:
prediction = grid_search_cat.best_estimator_.predict(test)
sub['Depression'] = prediction
sub.to_csv('submission.csv', index=False)

In [9]:
sub

,id,Depression
0,140700,0
1,140701,0
2,140702,0
3,140703,1
4,140704,0
...,...,...
93795,234495,0
93796,234496,1
93797,234497,0
93798,234498,1
